In [8]:
import xarray as xr
import rioxarray
import numpy as np
import os

# ==========================
# 配置
# ==========================
data_dir = r"E:\Project-yqr\GLEAM"
output_dir = r"E:\Project-yqr\new\data"
os.makedirs(output_dir, exist_ok=True)

years_to_extract = [2009, 2015, 2018]
var_name = "SMrz"   # GLEAM 变量名

# ==========================
# 遍历 nc 文件
# ==========================
for nc_file in os.listdir(data_dir):
    if not nc_file.endswith(".nc"):
        continue

    nc_path = os.path.join(data_dir, nc_file)
    print(f"正在处理文件: {nc_path}")

    # 打开数据
    ds = xr.open_dataset(nc_path, engine="netcdf4")
    if var_name not in ds:
        raise ValueError(f"变量 {var_name} 不在文件 {nc_file} 中，可用变量有: {list(ds.data_vars)}")

    da = ds[var_name]

    # ==========================
    # 按年份逐日累积
    # ==========================
    for year in years_to_extract:
        da_year = da.sel(time=da["time"].dt.year == year)
        if da_year.sizes["time"] == 0:
            continue

        print(f"  正在计算 {year} 年的年平均...")

        sum_data = None
        count = 0

        for t in da_year.time:
            daily = da_year.sel(time=t).values  # (lat, lon)
            if sum_data is None:
                sum_data = np.zeros_like(daily, dtype="float64")
            sum_data += daily
            count += 1

        year_mean = sum_data / count

        # 转为 DataArray
        da_mean = xr.DataArray(
            year_mean,
            dims=("lat", "lon"),
            coords={"lat": da["lat"], "lon": da["lon"]},
            attrs=da.attrs
        )

        # ⚡ 修改维度名以便写入栅格
        da_mean = da_mean.rename({"lat": "y", "lon": "x"})
        da_mean = da_mean.rio.write_crs("EPSG:4326")

        # 保存
        out_tif = os.path.join(output_dir, f"EU_SMrz_{year}.tif")
        da_mean.rio.to_raster(out_tif, compress="LZW")

        print(f"  已保存: {out_tif}")

    ds.close()


正在处理文件: E:\Project-yqr\GLEAM\SMrz_2009_GLEAM_v4.1a.nc
  正在计算 2009 年的年平均...
  已保存: E:\Project-yqr\new\data\EU_SMrz_2009.tif
正在处理文件: E:\Project-yqr\GLEAM\SMrz_2015_GLEAM_v4.1a.nc
  正在计算 2015 年的年平均...
  已保存: E:\Project-yqr\new\data\EU_SMrz_2015.tif
正在处理文件: E:\Project-yqr\GLEAM\SMrz_2018_GLEAM_v4.1a.nc
  正在计算 2018 年的年平均...
  已保存: E:\Project-yqr\new\data\EU_SMrz_2018.tif


In [9]:
import pandas as pd
import os

# ===============================
# 输入路径
# ===============================
data_dir = r"E:\Project-yqr\new\data"

gleam_file = os.path.join(data_dir, "gleam_smrz.csv")
SOC_2009_PATH = os.path.join(data_dir, "SOC_2009_filled.csv")
SOC_2015_PATH = os.path.join(data_dir, "SOC_2015_filled.csv")
SOC_2018_PATH = os.path.join(data_dir, "SOC_2018_filled.csv")

SOC_2009_OUT = SOC_2009_PATH.replace(".csv", "_SMrz_mapped.csv")
SOC_2015_OUT = SOC_2015_PATH.replace(".csv", "_SMrz_mapped.csv")
SOC_2018_OUT = SOC_2018_PATH.replace(".csv", "_SMrz_mapped.csv")

# ===============================
# 1. 读取 gleam 数据
# ===============================
gleam_df = pd.read_csv(gleam_file)

# 把 -9999 替换为空值
gleam_df = gleam_df.replace(-9999, pd.NA)

# ===============================
# 2. 定义一个函数用于合并
# ===============================
def merge_smrz(soc_path, year, out_path):
    soc_df = pd.read_csv(soc_path)

    # 按 ID 合并
    merge_df = soc_df.merge(
        gleam_df[["ID", f"SMrz_{year}"]],
        on="ID",
        how="left"
    )

    # 重命名列
    merge_df = merge_df.rename(columns={f"SMrz_{year}": "SMrz"})

    # 输出
    merge_df.to_csv(out_path, index=False)
    print(f"✅ 已保存: {out_path}")

# ===============================
# 3. 分别处理 2009 / 2015 / 2018
# ===============================
merge_smrz(SOC_2009_PATH, 2009, SOC_2009_OUT)
merge_smrz(SOC_2015_PATH, 2015, SOC_2015_OUT)
merge_smrz(SOC_2018_PATH, 2018, SOC_2018_OUT)


✅ 已保存: E:\Project-yqr\new\data\SOC_2009_filled_SMrz_mapped.csv
✅ 已保存: E:\Project-yqr\new\data\SOC_2015_filled_SMrz_mapped.csv
✅ 已保存: E:\Project-yqr\new\data\SOC_2018_filled_SMrz_mapped.csv


In [11]:
import os
import pandas as pd

# ===========================
# 输入路径
# ===========================
eci_dir = r"E:\Project-yqr\ECI_extracted"
data_dir = r"E:\Project-yqr\new\data"

SOC_2009_PATH = os.path.join(data_dir, "SOC_2009_filled.csv")
SOC_2015_PATH = os.path.join(data_dir, "SOC_2015_filled.csv")
SOC_2018_PATH = os.path.join(data_dir, "SOC_2018_filled.csv")

# 输出
SOC_2009_OUT = SOC_2009_PATH.replace(".csv", "_ECI_mapped.csv")
SOC_2015_OUT = SOC_2015_PATH.replace(".csv", "_ECI_mapped.csv")
SOC_2018_OUT = SOC_2018_PATH.replace(".csv", "_ECI_mapped.csv")

# ===========================
# 读取四个 ECI 指数
# ===========================
eci_files = {
    "cwd_periods": "number_of_cwd_periods_with_more_than_5days_per_time_period_2009_2018.csv",
    "cwd": "consecutive_wet_days_index_per_time_period_2009_2018.csv",
    "cdd_periods": "number_of_cdd_periods_with_more_than_5days_per_time_period_2009_2018.csv",
    "cdd": "consecutive_dry_days_index_per_time_period_2009_2018.csv",
}

eci_dfs = {}
for key, fname in eci_files.items():
    path = os.path.join(eci_dir, fname)
    df = pd.read_csv(path)
    eci_dfs[key] = df

# ===========================
# 合并函数
# ===========================
def merge_eci(soc_path, year, out_path):
    soc_df = pd.read_csv(soc_path)
    
    for key, df in eci_dfs.items():
        if str(year) not in df.columns:
            raise ValueError(f"{year} 不在 {key} 文件中，实际列: {list(df.columns)}")
        
        soc_df = soc_df.merge(
            df[["ID", str(year)]].rename(columns={str(year): key}),
            on="ID",
            how="left"
        )
    
    soc_df.to_csv(out_path, index=False)
    print(f"✅ 已保存 {out_path}")

# ===========================
# 分别处理三年
# ===========================
merge_eci(SOC_2009_PATH, 2009, SOC_2009_OUT)
merge_eci(SOC_2015_PATH, 2015, SOC_2015_OUT)
merge_eci(SOC_2018_PATH, 2018, SOC_2018_OUT)


✅ 已保存 E:\Project-yqr\new\data\SOC_2009_filled_ECI_mapped.csv
✅ 已保存 E:\Project-yqr\new\data\SOC_2015_filled_ECI_mapped.csv
✅ 已保存 E:\Project-yqr\new\data\SOC_2018_filled_ECI_mapped.csv


In [12]:
import pandas as pd

# ===============================
# 输入路径
# ===============================
clc_file = r"E:\Project-yqr\CLC_new\提取newclc.csv"
BD_PATH = r"E:\Project-yqr\new\data\BD_filled_select.csv"

# 输出路径
BD_OUT = BD_PATH.replace(".csv", "_CLC_mapped.csv")

# ===============================
# 1. 建立 CLC 映射关系 (111–523 → 1–44)
# ===============================
clc_mapping = {
    111: 1, 112: 2, 121: 3, 122: 4, 123: 5, 124: 6,
    131: 7, 132: 8, 133: 9,
    141: 10, 142: 11,
    211: 12, 212: 13, 213: 14,
    221: 15, 222: 16, 223: 17,
    231: 18,
    241: 19, 242: 20, 243: 21, 244: 22,
    311: 23, 312: 24, 313: 25,
    321: 26, 322: 27, 323: 28, 324: 29,
    331: 30, 332: 31, 333: 32, 334: 33, 335: 34,
    411: 35, 412: 36,
    421: 37, 422: 38, 423: 39,
    511: 40, 512: 41,
    521: 42, 522: 43, 523: 44,
    999: 999   # nodata 保留
}

# ===============================
# 2. 读取 CLC 数据并映射 2018
# ===============================
clc_df = pd.read_csv(clc_file)
clc_df["CLCACC_2018"] = clc_df["CLCACC_2018"].map(clc_mapping).fillna(999).astype(int)

# ===============================
# 3. 替换 BD 的 CLC 列
# ===============================
bd_df = pd.read_csv(BD_PATH)

merge_df = bd_df.merge(
    clc_df[["ID", "CLCACC_2018"]],
    on="ID",
    how="left"
)

merge_df["CLC"] = merge_df["CLCACC_2018"]
merge_df.drop(columns=["CLCACC_2018"], inplace=True)

# 保存
merge_df.to_csv(BD_OUT, index=False, encoding="utf-8-sig")
print(f"✅ 已保存: {BD_OUT}")


✅ 已保存: E:\Project-yqr\new\data\BD_filled_select_CLC_mapped.csv


In [ ]:
import os
import pandas as pd

# ===========================
# 输入路径
# ===========================
eci_dir = r"E:\Project-yqr\ECI_extracted"
data_dir = r"E:\Project-yqr\new\data"

SOC_2009_PATH = os.path.join(data_dir, "SOC_2009_filled.csv")
SOC_2015_PATH = os.path.join(data_dir, "SOC_2015_filled.csv")
SOC_2018_PATH = os.path.join(data_dir, "SOC_2018_filled.csv")

# 输出
SOC_2009_OUT = SOC_2009_PATH.replace(".csv", "_ECI_mapped.csv")
SOC_2015_OUT = SOC_2015_PATH.replace(".csv", "_ECI_mapped.csv")
SOC_2018_OUT = SOC_2018_PATH.replace(".csv", "_ECI_mapped.csv")

# ===========================
# 读取四个 ECI 指数
# ===========================
eci_files = {
    "cwd_periods": "number_of_cwd_periods_with_more_than_5days_per_time_period_2009_2018.csv",
    "cwd": "consecutive_wet_days_index_per_time_period_2009_2018.csv",
    "cdd_periods": "number_of_cdd_periods_with_more_than_5days_per_time_period_2009_2018.csv",
    "cdd": "consecutive_dry_days_index_per_time_period_2009_2018.csv",
}

eci_dfs = {}
for key, fname in eci_files.items():
    path = os.path.join(eci_dir, fname)
    df = pd.read_csv(path)
    eci_dfs[key] = df

# ===========================
# 合并函数
# ===========================
def merge_eci(soc_path, year, out_path):
    soc_df = pd.read_csv(soc_path)
    
    for key, df in eci_dfs.items():
        if str(year) not in df.columns:
            raise ValueError(f"{year} 不在 {key} 文件中，实际列: {list(df.columns)}")
        
        soc_df = soc_df.merge(
            df[["ID", str(year)]].rename(columns={str(year): key}),
            on="ID",
            how="left"
        )
    
    soc_df.to_csv(out_path, index=False)
    print(f"✅ 已保存 {out_path}")

# ===========================
# 分别处理三年
# ===========================
merge_eci(SOC_2009_PATH, 2009, SOC_2009_OUT)
merge_eci(SOC_2015_PATH, 2015, SOC_2015_OUT)
merge_eci(SOC_2018_PATH, 2018, SOC_2018_OUT)
